# MOGA-Phonons Experiment Dataframe Explorer

This notebook loads a pickled dataframe produced by `build_experiment_dataframe_updated.py` and provides quick visual diagnostics for a MOGA-Phonons experiment.

Designed for files such as:

```text
dataframes/dataframe000009.pkl
dataframes/dataframe000009.summary.json
```


## 1. Imports and paths

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Assumes this notebook is run from the repo root.
repo_root = Path.cwd()

experiment_id = "experiment000009"
df_path = repo_root / "dataframes" / "dataframe000009.pkl"
summary_path = repo_root / "dataframes" / "dataframe000009.summary.json"

print("Repo root:", repo_root)
print("Dataframe:", df_path)
print("Summary:", summary_path)


## 2. Load dataframe and summary

In [ ]:
df = pd.read_pickle(df_path)

if summary_path.exists():
    with open(summary_path, "r") as f:
        summary = json.load(f)
else:
    summary = {}

print("Rows:", len(df))
print("Columns:", len(df.columns))
display(pd.DataFrame(summary.items(), columns=["field", "value"]))
display(df.head())


## 3. Inspect dataframe schema

In [ ]:
schema_rows = []
for c in df.columns:
    s = df[c]
    try:
        n_unique = s.nunique(dropna=True)
    except TypeError:
        n_unique = None
    schema_rows.append({
        "column": c,
        "dtype": str(s.dtype),
        "non_null": int(s.notna().sum()),
        "n_unique": n_unique,
    })

schema = pd.DataFrame(schema_rows)
display(schema)


## 4. Basic experiment coverage

This checks how many rows are available per simulation and per generation.


In [ ]:
if "simulation_id" in df.columns:
    counts_by_sim = df.groupby("simulation_id").size().sort_index()
    display(counts_by_sim.describe())
    display(counts_by_sim.head())

    plt.figure(figsize=(10, 4))
    counts_by_sim.plot(kind="bar")
    plt.ylabel("Rows")
    plt.xlabel("Simulation")
    plt.title("Rows per simulation")
    plt.tight_layout()
    plt.show()

if "generation" in df.columns:
    counts_by_gen = df.groupby("generation").size().sort_index()
    display(counts_by_gen.describe())

    plt.figure(figsize=(8, 4))
    counts_by_gen.plot()
    plt.ylabel("Rows")
    plt.xlabel("Generation")
    plt.title("Rows per generation")
    plt.tight_layout()
    plt.show()


## 5. Stability overview

The builder adds `min_frequency`, `num_imaginary`, and `is_stable` when possible. These are useful first-pass metrics.


In [ ]:
stability_cols = [c for c in ["min_frequency", "max_frequency", "num_imaginary", "is_stable"] if c in df.columns]
display(df[stability_cols].describe(include="all") if stability_cols else "No stability columns found.")

if "is_stable" in df.columns:
    stable_counts = df["is_stable"].value_counts(dropna=False)
    display(stable_counts)
    plt.figure(figsize=(5, 4))
    stable_counts.plot(kind="bar")
    plt.ylabel("Count")
    plt.title("Stable vs unstable candidates")
    plt.tight_layout()
    plt.show()

if "min_frequency" in df.columns:
    plt.figure(figsize=(7, 4))
    df["min_frequency"].dropna().hist(bins=80)
    plt.xlabel("Minimum frequency")
    plt.ylabel("Count")
    plt.title("Distribution of minimum frequencies")
    plt.tight_layout()
    plt.show()

if "num_imaginary" in df.columns:
    plt.figure(figsize=(7, 4))
    df["num_imaginary"].dropna().hist(bins=80)
    plt.xlabel("Number of imaginary frequencies")
    plt.ylabel("Count")
    plt.title("Imaginary-mode count distribution")
    plt.tight_layout()
    plt.show()


## 6. Fitness distributions

In [ ]:
fitness_cols = [c for c in ["fitness1", "fitness2", "fitness3", "fitness_norm"] if c in df.columns]
display(df[fitness_cols].describe() if fitness_cols else "No fitness columns found.")

for col in fitness_cols:
    plt.figure(figsize=(7, 4))
    df[col].dropna().hist(bins=80)
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.title(f"Distribution of {col}")
    plt.tight_layout()
    plt.show()


## 7. Fitness evolution by generation

This shows whether the top-k candidates improve across generations.


In [ ]:
if "generation" in df.columns and "fitness_norm" in df.columns:
    gen_stats = df.groupby("generation")["fitness_norm"].agg(["mean", "max", "min", "std", "count"])
    display(gen_stats.head())

    plt.figure(figsize=(8, 4))
    gen_stats["max"].plot(label="max")
    gen_stats["mean"].plot(label="mean")
    gen_stats["min"].plot(label="min")
    plt.xlabel("Generation")
    plt.ylabel("Fitness norm")
    plt.title("Fitness norm evolution")
    plt.legend()
    plt.tight_layout()
    plt.show()

if "generation" in df.columns and "min_frequency" in df.columns:
    gen_minfreq = df.groupby("generation")["min_frequency"].agg(["mean", "max", "min"])
    plt.figure(figsize=(8, 4))
    gen_minfreq["max"].plot(label="max min-frequency")
    gen_minfreq["mean"].plot(label="mean min-frequency")
    plt.axhline(0.0, linestyle="--")
    plt.xlabel("Generation")
    plt.ylabel("Minimum frequency")
    plt.title("Stability evolution")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 8. Parameter sweep maps

These plots summarize behavior over lattice parameter and mass, if `a_val` and `mass` or `atomic_masses` are available.


In [ ]:
df_plot = df.copy()

if "mass" not in df_plot.columns and "atomic_masses" in df_plot.columns:
    def first_mass(x):
        if isinstance(x, (list, tuple, np.ndarray)):
            return float(x[0])
        return float(x)
    df_plot["mass"] = df_plot["atomic_masses"].map(first_mass)

metric = "fitness_norm" if "fitness_norm" in df_plot.columns else None
if metric and "a_val" in df_plot.columns and "mass" in df_plot.columns:
    pivot = df_plot.groupby(["mass", "a_val"])[metric].max().unstack("a_val")
    display(pivot)

    plt.figure(figsize=(8, 5))
    plt.imshow(pivot.values, aspect="auto", origin="lower")
    plt.colorbar(label=f"max {metric}")
    plt.xticks(range(len(pivot.columns)), [f"{x:.2f}" for x in pivot.columns], rotation=45)
    plt.yticks(range(len(pivot.index)), [f"{x:.1f}" for x in pivot.index])
    plt.xlabel("a_val")
    plt.ylabel("mass")
    plt.title(f"Parameter sweep map: max {metric}")
    plt.tight_layout()
    plt.show()

if "is_stable" in df_plot.columns and "a_val" in df_plot.columns and "mass" in df_plot.columns:
    stable_fraction = df_plot.groupby(["mass", "a_val"])["is_stable"].mean().unstack("a_val")
    display(stable_fraction)

    plt.figure(figsize=(8, 5))
    plt.imshow(stable_fraction.values, aspect="auto", origin="lower", vmin=0, vmax=1)
    plt.colorbar(label="stable fraction")
    plt.xticks(range(len(stable_fraction.columns)), [f"{x:.2f}" for x in stable_fraction.columns], rotation=45)
    plt.yticks(range(len(stable_fraction.index)), [f"{x:.1f}" for x in stable_fraction.index])
    plt.xlabel("a_val")
    plt.ylabel("mass")
    plt.title("Stable-candidate fraction across parameter sweep")
    plt.tight_layout()
    plt.show()


## 9. Force-constant relationships

Scatter plots of the reduced BvK parameters and their relation to fitness/stability.


In [ ]:
fc_cols = [c for c in ["alpha0", "alpha1", "beta1", "alpha2", "beta2"] if c in df.columns]
display(df[fc_cols].describe() if fc_cols else "No force-constant columns found.")

if len(fc_cols) >= 2:
    color_col = "fitness_norm" if "fitness_norm" in df.columns else None
    xcol, ycol = "alpha1", "beta1"
    if xcol in df.columns and ycol in df.columns:
        plt.figure(figsize=(6, 5))
        if color_col:
            sc = plt.scatter(df[xcol], df[ycol], c=df[color_col], s=8, alpha=0.6)
            plt.colorbar(sc, label=color_col)
        else:
            plt.scatter(df[xcol], df[ycol], s=8, alpha=0.6)
        plt.xlabel(xcol)
        plt.ylabel(ycol)
        plt.title(f"{ycol} vs {xcol}")
        plt.tight_layout()
        plt.show()

if "alpha0" in df.columns and "fitness_norm" in df.columns:
    plt.figure(figsize=(6, 5))
    plt.scatter(df["alpha0"], df["fitness_norm"], s=8, alpha=0.5)
    plt.xlabel("alpha0")
    plt.ylabel("fitness_norm")
    plt.title("Fitness vs alpha0")
    plt.tight_layout()
    plt.show()


## 10. Top candidates

This table lists the highest-ranking candidates according to `fitness_norm`, with stable candidates prioritized if available.


In [ ]:
preferred_cols = [
    "simulation_id", "generation", "solution_index", "rank",
    "a_val", "mass",
    "alpha0", "alpha1", "beta1", "alpha2", "beta2",
    "fitness1", "fitness2", "fitness3", "fitness_norm",
    "min_frequency", "num_imaginary", "is_stable",
    "yaml_path", "plot_path"
]
cols = [c for c in preferred_cols if c in df_plot.columns]

sort_cols = []
ascending = []
if "is_stable" in df_plot.columns:
    sort_cols.append("is_stable")
    ascending.append(False)
if "fitness_norm" in df_plot.columns:
    sort_cols.append("fitness_norm")
    ascending.append(False)
elif "min_frequency" in df_plot.columns:
    sort_cols.append("min_frequency")
    ascending.append(False)

top = df_plot.sort_values(sort_cols, ascending=ascending).head(25) if sort_cols else df_plot.head(25)
display(top[cols])


## 11. Save filtered subsets

Useful for making small curated datasets for plotting, manuscript figures, or Zenodo releases.


In [ ]:
output_dir = repo_root / "dataframes" / "subsets"
output_dir.mkdir(parents=True, exist_ok=True)

if "is_stable" in df_plot.columns:
    stable_df = df_plot[df_plot["is_stable"] == True].copy()
    stable_path = output_dir / f"{experiment_id}_stable_candidates.pkl"
    stable_df.to_pickle(stable_path)
    print("Saved stable candidates:", stable_path, "rows =", len(stable_df))

if "fitness_norm" in df_plot.columns:
    top_path = output_dir / f"{experiment_id}_top100_by_fitness_norm.pkl"
    df_plot.sort_values("fitness_norm", ascending=False).head(100).to_pickle(top_path)
    print("Saved top 100:", top_path)
